In [1]:
#1. Importing Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
#2. Read and prepare data
# Read dataset
df = pd.read_csv("prep.csv")

# One-hot encode categorical variables
df = pd.get_dummies(df, drop_first=True)

# Split features and target
X = df.drop("classification_yes", axis=1)
y = df["classification_yes"]

In [3]:
#3. Load all classifier functions

def split_and_scale(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=0
    )
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    return X_train, X_test, y_train, y_test


def evaluate_accuracy(X, y):
    X_train, X_test, y_train, y_test = split_and_scale(X, y)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds)

In [4]:
#4. Logic for Forward Selection algorithm

all_features = list(X.columns)
selected_features = []
remaining_features = all_features.copy()

results = []
best_score = 0

while len(remaining_features) > 0:
    scores_with_candidates = []

    for feature in remaining_features:
        candidate_features = selected_features + [feature]
        candidate_X = X[candidate_features]
        score = evaluate_accuracy(candidate_X, y)
        scores_with_candidates.append((feature, score))

    # Pick the feature that gives the highest accuracy
    best_feature, best_candidate_score = max(
        scores_with_candidates, key=lambda x: x[1]
    )

    # Stop if no improvement
    if best_candidate_score <= best_score:
        break

    selected_features.append(best_feature)
    remaining_features.remove(best_feature)
    best_score = best_candidate_score

    results.append((len(selected_features), best_score, selected_features.copy()))

In [5]:
## 5 Create the result dataframe and display Best Feature

results_df = pd.DataFrame([(r[0], r[1]) for r in results],columns=["Num_features", "Accuracy"])

best_row = results_df.loc[results_df["Accuracy"].idxmax()]
best_feature_count = int(best_row["Num_features"])

best_features = results[results_df["Accuracy"].idxmax()][2]

print("Best number of features:", best_feature_count)

Best number of features: 4


In [9]:
results_df

,Num_features,Accuracy
0,1,0.91
1,2,0.95
2,3,0.99
3,4,1.00


In [6]:
#6. Selected features
print("Selected Features (Forward Selection):")
for f in best_features:
    print(f)

Selected Features (Forward Selection):
hrmo
sg_d
sg_c
bu
